# inference

> `Decider`: typed answers in the Jev schema from a trained model, in memory or loaded from an export folder that Laya's runtimes can read too.

In [ ]:
#| default_exp inference

In [ ]:
#| export
import copy, json, math, os, shutil, time, warnings
from pathlib import Path

import numpy as np
import torch
from fastcore.basics import patch, store_attr
from transformers import AutoConfig, AutoModel, AutoTokenizer

import sysone
from sysone.core import *
from sysone.template import *
from sysone.models import *
from sysone.learner import Learner

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import logging, tempfile
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

`Decider.predict(state, questions)` returns the Jev answer schema, so a model trained with sysone drops into anything written for Jev or Laya. It builds one row per question with the saved template and transforms, batches them (padding to the longest, markers padded and masked), runs one forward pass, divides each row's logits by its bucket's temperature, and fills the schema. Several questions are several rows of one batch; the cost grows with questions, never with a second pass.

```{mermaid}
flowchart LR
    IN["state + questions"] --> TF["transforms,<br/>once per state"]
    TF --> RB["RowBuilder:<br/>one row per question"]
    RB --> BA["batches of<br/>batch_size rows"]
    BA --> FW["one forward pass<br/>per batch"]
    FW --> TE["logits ÷ the bucket's<br/>temperature"]
    TE --> SM["softmax over each<br/>row's options"]
    SM --> FA["fill_answer:<br/>the Jev schema"]
```

A small trained learner to work with, the fixture encoder with Laya's head:

In [ ]:
from sysone.data import TypedDecisions

In [ ]:
FIX = "fixtures/tiny-modernbert"
recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25, calib=0.25)
torch.manual_seed(0)
learn = sysone.learner.decision_learner(data, FIX, train="full", preset="cpu", loss="soft_ce")
learn.fit_one_cycle(2, lr=2e-3)
learn.calibrate(min_rows=4)

## Decider

In [ ]:
#| export
def default_device() -> torch.device:
    "CUDA, then MPS, then CPU"
    if torch.cuda.is_available(): return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

class Decider:
    "Typed answers from a decision model and the row builder it was trained with"
    def __init__(self,
                 model:EncoderDecisionModel, # encoder and head
                 builder:RowBuilder, # the template and transforms of training
                 temperatures:dict|None=None, # per type and option-count bucket
                 device=None, # where to run; the model stays where it is by default (`Decider.load` picks CUDA, MPS, CPU)
                 dtype=None, # autocast dtype; fp16 on CUDA, none (fp32) elsewhere
                 batch_size:int=64, # rows per forward pass
                 digits:int|None=4, # rounding of reported numbers, as Laya rounds
                 name:str="sysone", # the model name reported by the HTTP API
                 chunk:int|None=None, # score choices wider than this coarse to fine, in chunks of at most `chunk` options
                 meta:dict|None=None, # the export's sysone.json, when loaded from one
                 act_threshold:float|dict|None=None, # act where the act score ≥ this, per type and bucket (fitted by `learn.calibrate`)
                 act_score:str="head"): # the act score: the act head's p(act) ('head') or the calibrated top probability ('confidence')
        store_attr(but="model,device")
        self.device = torch.device(device) if device is not None else next(model.parameters()).device
        self.model = model.to(self.device).eval()
        if self.dtype is None and self.device.type == "cuda": self.dtype = torch.float16
        self.temperatures = temperatures or {}

    def __repr__(self):
        return f"Decider({self.name} · {self.builder.template.name or 'custom'} template · {self.device} · {len(self.temperatures)} temperatures)"

### Answering

`_rows` runs the transforms once per state and builds the rows from the transformed record, so a transform that changes a question, such as a `Shortlist`, is seen by the answer too. `_scores` runs the rows in chunks of `batch_size` and returns their logits and, from a model with an act head, each row's p(act) (`_logits` keeps only the logits); `_probs` (and `_probs_record`, for a record already transformed) turns each row's logits into option probabilities at its bucket's temperature; `_answer` fills the schema.

In [ ]:
#| export
@patch
def _rows(self:Decider, state, questions):
    rec = apply_tfms(self.builder.tfms, load_record({"state": state, "questions": questions}), train=False)
    return rec, self.builder.build_record(rec, tfms=False)

@patch
@torch.no_grad()
def _scores(self:Decider, rows) -> tuple:
    "(logits (rows, K), p(act) per row or None)"
    out, acts, pad = [], [], self.builder.tok.pad_token_id or 0
    for i in range(0, len(rows), self.batch_size):
        b = collate(rows[i:i + self.batch_size], pad).to(self.device)
        for k in ("target", "label", "answerable"): b.pop(k, None)
        ctx = torch.autocast(self.device.type, dtype=self.dtype) if self.dtype is not None else torch.autocast("cpu", enabled=False)
        with ctx: res = self.model(**b)
        z, a = res if isinstance(res, tuple) else (res, None)
        out.append(z.float().cpu().numpy())
        if a is not None: acts.append(act_probability(a).cpu().numpy())
    K = max(z.shape[1] for z in out)
    return np.concatenate([np.pad(z, ((0, 0), (0, K - z.shape[1])), constant_values=-1e4) for z in out]), np.concatenate(acts) if acts else None

@patch
def _logits(self:Decider, rows) -> np.ndarray: return self._scores(rows)[0]

@patch
def _softmax(self:Decider, q:Question, z) -> np.ndarray:
    z = z[:q.k].astype(np.float64) / lookup_temperature(self.temperatures, q.qtype, q.k)
    p = np.exp(z - z.max())
    return p / p.sum()

@patch
def _probs(self:Decider, state, questions:dict) -> tuple:
    "`{qid: (question, probabilities, options a shortlist dropped, p(act) or None)}` for one state, and its number of input tokens"
    return self._probs_record(apply_tfms(self.builder.tfms, load_record({"state": state, "questions": questions}, parse_state=False)))

@patch
def _probs_record(self:Decider, rec:dict) -> tuple:
    "`_probs` for a record whose transforms have run"
    rows = self.builder.build_record(rec, tfms=False)
    if not rows: return {}, 0
    z, act = self._scores(rows)
    qs, dropped = as_questions(rec["questions"]), rec.get("dropped") or {}
    out = {r.meta["qid"]: (qs[r.meta["qid"]], self._softmax(qs[r.meta["qid"]], z[i]), dropped.get(r.meta["qid"]),
                           float(act[i]) if act is not None else None) for i, r in enumerate(rows)}
    return out, sum(r.n_tokens for r in rows)

@patch
def _answer(self:Decider, q:Question, p, dropped=None, act=None) -> Answer:
    if self.act_score == "confidence" and self.act_threshold is not None: act = float(np.max(np.asarray(p, dtype=np.float64)[:q.k]))
    a = fill_answer(q, p, self.digits, act=act, act_threshold=self.act_threshold)
    if dropped:   # options a shortlist removed are reported with probability 0
        a["probabilities"] = {kk: a["probabilities"].get(kk, 0.0) for kk in list(a["probabilities"]) + list(dropped)}
        a["shortlisted"] = True
    return a

In [ ]:
#| export
@patch
def answer(self:Decider, state, questions:dict) -> tuple:
    "Answers in the Jev schema for one state, and the number of input tokens they took"
    if not questions: return {}, 0
    rec = apply_tfms(self.builder.tfms, load_record({"state": state, "questions": questions}, parse_state=False))
    if self._wide(rec["questions"]): return self._answer_wide(rec)        # wide after the transforms have had their say
    probs, n = self._probs_record(rec)
    return {qid: self._answer(q, p, d, a) for qid, (q, p, d, a) in probs.items()}, n

@patch
def predict(self:Decider, state, questions:dict) -> dict:
    "Answers `{qid: answer}` in the Jev schema for one state"
    return self.answer(state, questions)[0]

@patch
def predict_batch(self:Decider, states:list, questions:dict) -> list:
    "Answers for many states, their rows sharing forward passes"
    if not states: return []
    if not questions: return [{} for _ in states]
    if self._wide(questions): return [self.predict(s, questions) for s in states]
    built = [self._rows(s, questions) for s in states]
    flat = [r for _, rows in built for r in rows]
    (z, act), out, i = self._scores(flat), [], 0
    for rec, rows in built:
        qs, dropped = as_questions(rec["questions"]), rec.get("dropped") or {}
        out.append({r.meta["qid"]: self._answer(qs[r.meta["qid"]], self._softmax(qs[r.meta["qid"]], z[i + j]), dropped.get(r.meta["qid"]),
                                                float(act[i + j]) if act is not None else None)
                    for j, r in enumerate(rows)})
        i += len(rows)
    return out

### Wide option sets

Options share `head_max_len`, which is Laya's known weakness above about twenty options (0.425 on Banking77's 77 intents). With `chunk=n` a wider choice is scored coarse to fine, the pattern the laya-browser servers use: its options are dealt into interleaved chunks of at most n, each chunk is a row of a first pass, and the chunk winners meet in one row of a second pass. An option's probability is its chunk winner's probability in the final row times its own within its chunk, which sums to one over all the options. The transforms run first, once, so a question is chunked as they leave it (a `Shortlist` may have made it narrow enough already).

In [ ]:
#| export
@patch
def _wide(self:Decider, questions:dict) -> dict:
    "For each choice question with more options than `chunk`, its keys dealt into interleaved chunks"
    if not getattr(self, "chunk", None): return {}
    out = {}
    for qid, q in as_questions(questions).items():
        if q.type == "choice" and q.k > self.chunk:
            n = math.ceil(q.k / self.chunk)
            out[qid] = [q.keys[c::n] for c in range(n)]
    return out

@patch
def _answer_wide(self:Decider, rec:dict) -> tuple:
    "Answers for a transformed record with wide questions: a first pass over interleaved chunks, a second over the winners"
    questions = rec["questions"]
    wide, qs, dropped = self._wide(questions), as_questions(questions), rec.get("dropped") or {}
    sub = lambda q, keys: {**q.to_dict(), "criteria": {k: q.criteria[k] for k in keys}}
    first = {qid: questions[qid] for qid in qs if qid not in wide}
    first |= {f"{qid}#{c}": sub(qs[qid], ks) for qid, chunks in wide.items() for c, ks in enumerate(chunks)}
    p1, n1 = self._probs_record({**rec, "questions": first, "dropped": {}})
    top = lambda q, p: q.keys[int(np.argmax(p))]
    winners = {qid: [top(*p1[f"{qid}#{c}"][:2]) for c in range(len(chunks))] for qid, chunks in wide.items()}
    p2, n2 = self._probs_record({**rec, "questions": {qid: sub(qs[qid], ws) for qid, ws in winners.items()}, "dropped": {}})
    answers = {qid: self._answer(q, p, dropped.get(qid), a) for qid, (q, p, d, a) in p1.items() if qid in qs}
    for qid, chunks in wide.items():
        fq, fp, _, fa = p2[qid]
        final = dict(zip(fq.keys, fp))
        prob = {}
        for c in range(len(chunks)):
            cq, cp, _, _ = p1[f"{qid}#{c}"]
            prob |= {k: final[winners[qid][c]] * v for k, v in zip(cq.keys, cp)}
        answers[qid] = self._answer(qs[qid], np.array([prob[k] for k in qs[qid].keys]), dropped.get(qid), fa)
        answers[qid]["chunks"] = len(chunks)
    return {qid: answers[qid] for qid in qs}, n1 + n2

A learner answers from the model in memory, before any export: `learn.decider()` wraps its model, builder and temperatures in a `Decider`, and `learn.predict` is a shortcut.

In [ ]:
rec = recs[1]
ans = learn.predict(rec["state"], rec["questions"])
ans

In [ ]:
test_eq(list(ans), list(rec["questions"]))
test_eq({a["type"] for a in ans.values()}, {"choice", "noul", "score"})
for qid, a in ans.items():
    if a["type"] != "noul": test_close(sum(a["probabilities"].values()), 1.0, eps=1e-3)
test_eq(learn.decider().predict_batch([rec["state"], recs[2]["state"]], rec["questions"])[0], ans)
test_eq(learn.decider().predict_batch([], rec["questions"]), [])

Questions whose definitions cannot be answered fail with the question's name, as Laya's runtime does:

In [ ]:
jev = learn.decider()
test_fail(lambda: jev.predict("x", {"q": {"type": "choice", "instructions": "Pick", "criteria": []}}), contains="'q'")
test_eq(jev.predict("x", {}), {})

Coarse to fine on a 40-option question with chunks of 8: five first-pass rows and one final row, and the probabilities still sum to one:

In [ ]:
wide_q = {"intent": {"type": "choice", "instructions": "What does the customer want?", "criteria": [f"intent {i}" for i in range(40)]}}
wide_jev = learn.decider(chunk=8)
aw = wide_jev.predict("my card has not arrived yet", wide_q)["intent"]
test_eq((len(aw["probabilities"]), aw["chunks"]), (40, 5))
test_close(sum(aw["probabilities"].values()), 1.0, eps=1e-3)
test_eq(wide_jev.predict(rec["state"], rec["questions"]), jev.predict(rec["state"], rec["questions"]))   # narrow questions: unchanged
aw["choice"], aw["answer_confidence"]

## Export

`learn.export(path)` writes one folder that sysone and, when the template allows, Laya can read:

```
my-jev/
  sysone.json         template, transforms, head shape, temperatures, encoder reference, regime, metrics
  head.safetensors    type embedding, head layers, scorer
  tokenizer/          (or processor/ for multimodal encoders)
  streams/<name>/     each side stream's tokenizer
  encoder/            the encoder's config, and its weights when they are bundled (a stream encoder: text/, sides/<name>/ and its own file)
  rl_agent_config.json + model.safetensors   Laya's format, when the template is Laya's
```

- Adapters are merged at export, so a LoRA or MiCA run ships as a plain encoder; `merge=False` keeps the adapter in `adapter/` for further training.
- A head-only run on a Hub encoder stays small: the export holds the head and the encoder's id and revision, and `Decider.load` pulls the encoder. `bundle=True` copies the weights in; a local encoder is always bundled.
- When the template, the head and the transforms are Laya's, `laya="auto"` also writes `rl_agent_config.json` and a single `model.safetensors` with Laya's parameter names, which `laya.Agent`, `laya[serve]` and `laya-mlx` load as they are. The encoder weights then live in that file only, and `encoder/` holds the config.

What decides the encoder's part of an export:

```{mermaid}
flowchart TB
    X["learn.export(path)"] --> T{"did the encoder train?<br/>(the regime is not head, or it trained before a freeze)"}
    T -- yes --> BUN["bundled: its weights are written"]
    T -- no --> LOC{"a local encoder?"}
    LOC -- yes --> BUN
    LOC -- no --> REF["the encoder's id and revision only<br/>(bundle=True copies the weights in)"]
    BUN --> AD{"adapters, merge=False?"}
    AD -- yes --> KEEP["the base weights, and adapter/ beside them"]
    AD -- no --> LY{"Laya's template, head<br/>and no inference transforms?"}
    LY -- yes --> LF["rl_agent_config.json + model.safetensors:<br/>Laya's format, the adapters merged"]
    LY -- no --> ENC["encoder/: a plain encoder, the adapters merged"]
```

In [ ]:
#| export
def _encoder_ref(model, spec) -> dict:
    enc = model.encoder.get_base_model() if hasattr(model.encoder, "get_base_model") else model.encoder
    ref = spec.to_json() if spec is not None else {"id": getattr(enc.config, "_name_or_path", None), "revision": None,
                                                   "source": "transformers", "hidden_size": model.head.width,
                                                   "family": enc.config.model_type, "modality": "text"}
    ref["dtype"] = str(next(enc.parameters()).dtype).removeprefix("torch.")
    return ref

def _laya_config(model, template, temperatures, name) -> dict:
    t = temperatures or {}
    return {"encoder": model.spec.id if model.spec is not None else "answerdotai/ModernBERT-large",
            "head_layers": model.head.layers, "max_len": template.max_len, "head_max_len": template.head_max_len,
            "temperature": [clamp_temperature(t.get(n, 1.0)) for n in ("choice", "score", "noul")],
            "temperature_by_options": {k: v for k, v in t.items() if ":" in k}, "model_name": name, "exported_by": "sysone"}

def export_learner(learn:Learner, path, merge:bool=True, bundle:bool|None=None, laya="auto", dtype=None, name:str|None=None) -> Path:
    "Write a learner's model, template, transforms and temperatures to `path` (what `learn.export` calls)"
    self = learn
    path, model, b = Path(path), self.model, self.data.builder
    path.mkdir(parents=True, exist_ok=True)
    spec, ref = model.spec, _encoder_ref(model, model.spec)
    trained_encoder = model.regime != "head" or getattr(model, "encoder_trained", False)   # trained, then frozen again
    local_encoder = ref["id"] is not None and Path(ref["id"]).exists()
    bundle = (trained_encoder or local_encoder) if bundle is None else bundle
    if trained_encoder and not bundle: raise ValueError("the encoder was trained, so its weights must be exported (bundle=True)")
    write_laya = laya is True or (laya == "auto" and bundle and merge and is_laya_compatible(model, b.template, b.tfms, b.tok))
    if laya is True and not is_laya_compatible(model, b.template, b.tfms, b.tok):
        raise ValueError("Laya's format needs Laya's template (and budgets), Laya's head and no inference transforms")
    # tokenizer or processor
    (b.processor or b.tok).save_pretrained(path / ("processor" if b.processor is not None else "tokenizer"))
    for name, s in b.streams.items(): s.tokenizer.save_pretrained(path / "streams" / name)
    # encoder
    enc_dir = path / "encoder"
    keep_adapter = not merge and hasattr(model.encoder, "peft_config")
    if write_laya:
        merged_encoder(model).config.save_pretrained(enc_dir); weights = "laya"
        from safetensors.torch import save_file
        save_file(laya_state_dict(model, self.temperatures, dtype=dtype), str(path / "model.safetensors"))
        (path / "rl_agent_config.json").write_text(json.dumps(_laya_config(model, b.template, self.temperatures, name or path.name), indent=2))
    elif keep_adapter:   # the base encoder's own weights, and the adapter beside them
        copy.deepcopy(model.encoder).unload().save_pretrained(enc_dir)
        model.encoder.save_pretrained(path / "adapter"); weights = "adapter"
    elif bundle:
        enc = merged_encoder(model)
        (copy.deepcopy(enc).to(dtype) if dtype is not None else enc).save_pretrained(enc_dir); weights = "encoder"
    else: weights = "hub"
    # head
    from safetensors.torch import save_file
    sd = {k: (v.to(dtype) if dtype is not None and v.is_floating_point() else v).detach().cpu().contiguous()
          for k, v in model.head.state_dict().items()}
    save_file(sd, str(path / "head.safetensors"))
    meta = {"sysone_version": sysone.__version__, "backend": "sysone", "name": name or path.name,
            "encoder": ref | {"weights": weights}, **b.to_json(),
            "head": model.head.config() | {"regime": model.regime},
            "temperature": self.temperatures, "act_threshold": getattr(self, "act_threshold", None),
            "act_score": getattr(self, "act_score", None), "question_types": list(QTYPES),
            "metrics": self.recorder.metrics[-1] if self.recorder.metrics else None}
    (path / "sysone.json").write_text(json.dumps(meta, indent=2, default=str))
    return path

In [ ]:
out = Path(tempfile.mkdtemp()) / "my-jev"
learn.export(out)
sorted(p.name for p in out.iterdir()), sorted(p.name for p in (out / "encoder").iterdir())

In [ ]:
meta = json.loads((out / "sysone.json").read_text())
test_eq((meta["encoder"]["weights"], meta["head"]["regime"], meta["template"]["row"]), ("laya", "full", RowTemplate.preset("laya").row))
test_eq(json.loads((out / "rl_agent_config.json").read_text())["head_layers"], 2)
{k: meta[k] for k in ("encoder", "head", "temperature")}

## Decider.load

`Decider.load(path)` reads a folder (or a Hub repo, with a subfolder written `owner/repo/subfolder`) and picks the backend: a `sysone.json` loads the sysone head over its encoder, a bare `rl_agent_config.json` loads a Laya checkpoint as it is, and a GLiNER2 checkpoint loads the GLiNER2 backend (`sysone.gliner`, which needs the `gliner2` package). Backends register in `DECIDER_BACKENDS`.

```{mermaid}
flowchart LR
    P["Decider.load(folder, Hub id,<br/>or owner/repo/subfolder)"] --> S{"sysone.json?"}
    S -- yes --> SY["the sysone head over its encoder"]
    S -- no --> L{"rl_agent_config.json?"}
    L -- yes --> LA["a Laya checkpoint, as it is"]
    L -- no --> G{"a GLiNER2 checkpoint?"}
    G -- yes --> GL["the GLiNER2 backend"]
    G -- no --> E["ValueError naming the three formats"]
```

In [ ]:
#| export
def _load_sysone(path:Path, device=None, **kw) -> Decider:
    meta = json.loads((path / "sysone.json").read_text())
    ref, weights = meta["encoder"], meta["encoder"].get("weights", "hub")
    dt = getattr(torch, ref.get("dtype") or "float32")          # the dtype the head was trained against
    if (path / "processor").exists():
        from transformers import AutoProcessor
        tok = AutoProcessor.from_pretrained(path / "processor")
    else: tok = AutoTokenizer.from_pretrained(path / "tokenizer")
    stream_toks = {p.name: AutoTokenizer.from_pretrained(p) for p in sorted((path / "streams").iterdir())} if (path / "streams").exists() else {}
    base = lambda d: load_stream_encoder(d, dt) if (d / "stream_encoder.json").exists() else AutoModel.from_pretrained(d, dtype=dt)
    if weights == "hub":
        if ref["id"] is None or (ref["id"].startswith(("/", ".")) and not Path(ref["id"]).exists()):
            raise FileNotFoundError(f"the export references encoder {ref['id']!r}, which is not available; export with bundle=True")
        if ref.get("source") in (None, "transformers"): enc = AutoModel.from_pretrained(ref["id"], revision=ref.get("revision"), dtype=dt)
        else: enc = EncoderSpec.from_pretrained(ref["id"], revision=ref.get("revision"), source=ref["source"]).load_encoder(dtype=dt)
    elif weights == "encoder": enc = base(path / "encoder")
    elif weights == "adapter":
        from peft import PeftModel
        enc = PeftModel.from_pretrained(base(path / "encoder"), path / "adapter")
    else:   # "laya": the encoder's tensors live in Laya's single weights file
        from safetensors.torch import load_file
        enc = AutoModel.from_config(AutoConfig.from_pretrained(path / "encoder"), dtype=dt)
        sd = {k[len("encoder."):]: v.to(dt) for k, v in load_file(str(path / "model.safetensors")).items() if k.startswith("encoder.")}
        enc.load_state_dict(sd, strict=False)
    h = meta["head"]
    head = DecisionHead.from_config(h)
    from safetensors.torch import load_file
    head.load_state_dict({k: v.float() for k, v in load_file(str(path / "head.safetensors")).items()})
    model = EncoderDecisionModel(enc, head)
    model.regime = h.get("regime", "full")
    builder = RowBuilder.from_json({"template": meta["template"], "tfms": meta.get("tfms", []), "streams": meta.get("streams")}, tok, stream_toks)
    return Decider(model, builder, meta.get("temperature") or {}, device=device, name=meta.get("name", "sysone"), meta=meta,
                   **({"act_threshold": meta.get("act_threshold"), "act_score": meta.get("act_score") or "head"} | kw))

def _load_laya(path:Path, device=None, **kw) -> Decider:
    model, tok, template, temps = load_laya(str(path))
    return Decider(model, RowBuilder(tok, template), temps, device=device, name="laya", **kw)

def _is_gliner(path:Path) -> bool:
    if (path / "sysone_gliner.json").exists(): return True
    cfg = path / "config.json"
    if not cfg.exists() or not (path / "encoder_config").exists(): return False
    c = json.loads(cfg.read_text())
    return c.get("model_type") == "extractor" or "architecture" in c

def _load_gliner(path:Path, device=None, **kw):
    from sysone.gliner import GlinerDecider
    return GlinerDecider.load(str(path), device=device, **kw)

DECIDER_BACKENDS = [("sysone", lambda p: (p / "sysone.json").exists(), _load_sysone),
                    ("laya", lambda p: (p / "rl_agent_config.json").exists(), _load_laya),
                    ("gliner2", _is_gliner, _load_gliner)]

def _local(path, subfolder=None, revision=None) -> Path:
    p = Path(path).expanduser()
    if p.exists(): return p / (subfolder or "")
    if str(path).startswith(("/", "./", "../", "~")): raise FileNotFoundError(f"no such folder: {path}")
    from huggingface_hub import snapshot_download
    pre = f"{subfolder}/" if subfolder else ""
    return Path(snapshot_download(str(path), revision=revision, allow_patterns=[pre + x for x in
        ("*.json", "*.safetensors", "tokenizer/*", "processor/*", "encoder/*", "encoder_config/*", "adapter/*", "*.txt", "*.model")])) / (subfolder or "")

@patch(cls_method=True)
def load(cls:Decider, path, device=None, subfolder:str|None=None, revision:str|None=None, **kw):
    "The decider stored at `path` (a folder, a Hub repo, or `owner/repo/subfolder`): sysone, Laya or GLiNER2"
    if subfolder is None: path, subfolder = split_repo(str(path))
    p = _local(path, subfolder, revision)
    device = device if device is not None else default_device()
    for name, detect, load in DECIDER_BACKENDS:
        if detect(p): return load(p, device=device, **kw)
    raise ValueError(f"{path}: neither a sysone export (sysone.json), a Laya checkpoint (rl_agent_config.json) nor a GLiNER2 checkpoint")

The exported folder gives the same answers as the model in memory:

In [ ]:
jev = Decider.load(out, device="cpu")
jev

In [ ]:
for rc in recs[:6]:
    test_eq(jev.predict(rc["state"], rc["questions"]), learn.predict(rc["state"], rc["questions"], device="cpu"))

### Laya compatibility

The same folder loads in `laya.Agent`, Laya's own runtime, which rebuilds the rows with its own code and reads the temperatures from `rl_agent_config.json`. Its answers match sysone's on every fixture case (Laya adds an `action` field, from an action head sysone does not train; it is written as zeros, so Laya reports an action probability of 1):

In [ ]:
import laya

In [ ]:
agent = laya.Agent(str(out), device="cpu")
for rc in recs:
    ours, theirs = jev.predict(rc["state"], rc["questions"]), agent.predict(rc["state"], rc["questions"])["answers"]
    for qid in ours:
        a, b = ours[qid], theirs[qid]
        test_eq(a["type"], b["type"])
        for key in ("choice", "score", "noul", "confidence", "answer_confidence"):
            if key in a: test_close(a[key], b[key], eps=2e-4) if not isinstance(a[key], str) else test_eq(a[key], b[key])
        if "probabilities" in a: test_close(list(a["probabilities"].values()), list(b["probabilities"].values()), eps=2e-4)
theirs[qid]

A head-only run on a Hub encoder exports only the head and the encoder's reference; here the fixture encoder is local, so it is bundled, and `laya=False` skips Laya's format:

In [ ]:
torch.manual_seed(0)
lh = sysone.learner.decision_learner(data, FIX, train="head", preset="cpu", loss="soft_ce")
lh.fit(1, lr=1e-3)
out2 = lh.export(Path(tempfile.mkdtemp()) / "head-only", laya=False)
test_eq(sorted(p.name for p in out2.iterdir()), ["encoder", "head.safetensors", "sysone.json", "tokenizer"])
test_eq(json.loads((out2 / "sysone.json").read_text())["encoder"]["weights"], "encoder")
test_eq(Decider.load(out2, device="cpu").predict(rec["state"], rec["questions"]), lh.predict(rec["state"], rec["questions"]))

::: {.callout-note title="Finding: an export after freezing dropped the trained encoder"}
The first export looked only at the current regime: after `unfreeze`, `fit` and `freeze`, the regime reads `head`, so it wrote the head and a reference to the Hub encoder, and the encoder the head had been trained on was lost. `_run` now marks the model when its encoder trains, and the export bundles the encoder whenever that mark is set. The same review found the dtype problem: a checkpoint stored in bf16 was served in bf16 though its head was trained on float32 features; the export now records the dtype the encoder trained in, and the loader restores it.
:::

An encoder that trained is exported even when the regime is back to `head` (unfreeze, fit, freeze), since the head was trained on it; and the export records the dtype the encoder trained in, which the loader restores, rather than the dtype a checkpoint happens to be stored in:

In [ ]:
torch.manual_seed(0)
lf = sysone.learner.decision_learner(data, FIX, train="head", preset="cpu", loss="soft_ce")
lf.unfreeze(); lf.fit(1, lr=1e-3); lf.freeze()
lf.model.spec.id = "someone/an-encoder-on-the-hub"          # as if it came from the Hub
outf = lf.export(Path(tempfile.mkdtemp()) / "refrozen", laya=False)
test_eq(json.loads((outf / "sysone.json").read_text())["encoder"]["weights"], "encoder")
test_eq(Decider.load(outf, device="cpu").predict(rec["state"], rec["questions"]), lf.predict(rec["state"], rec["questions"]))

bf = Path(tempfile.mkdtemp()) / "bf16-encoder"
AutoModel.from_pretrained(FIX).to(torch.bfloat16).save_pretrained(bf); AutoTokenizer.from_pretrained(FIX).save_pretrained(bf)
lb = sysone.learner.decision_learner(data, str(bf), train="head", preset="cpu", loss="soft_ce")
test_eq(next(lb.model.encoder.parameters()).dtype, torch.float32)
outb = lb.export(Path(tempfile.mkdtemp()) / "bf16", bundle=False, laya=False)
test_eq(json.loads((outb / "sysone.json").read_text())["encoder"]["dtype"], "float32")
test_eq(next(Decider.load(outb, device="cpu").model.encoder.parameters()).dtype, torch.float32)

A wide question whose options a transform thins out still answers, the removed options at probability 0:

In [ ]:
from sysone.template import Transform, transform

In [ ]:
@transform("DropEven")
class DropEven(Transform):
    def __call__(self, record):   # drops "intent 0", "intent 2", ...: every option whose number is even, and says so
        keep = lambda c: int(str(c).split()[-1]) % 2 == 1
        qs = {q: {**d, "criteria": [c for c in d["criteria"] if keep(c)]} for q, d in record["questions"].items()}
        dropped = {q: [c for c in d["criteria"] if not keep(c)] for q, d in record["questions"].items()}
        return {**record, "questions": qs, "dropped": dropped}
thin = Decider(learn.model, RowBuilder(learn.data.builder.tok, learn.data.builder.template, [DropEven()]), chunk=4)
aw2 = thin.predict("my card has not arrived", wide_q)["intent"]
test_eq((len(aw2["probabilities"]), aw2["probabilities"]["intent 0"], aw2["shortlisted"], aw2["chunks"]), (40, 0.0, True, 5))
test_close(sum(aw2["probabilities"].values()), 1.0, eps=1e-3)

A LoRA run can keep its adapter, for further training, and still load:

In [ ]:
torch.manual_seed(0)
ll = sysone.learner.decision_learner(data, FIX, train="lora", preset="cpu", loss="soft_ce")
ll.fit(1, lr=5e-3)
out3 = ll.export(Path(tempfile.mkdtemp()) / "lora", merge=False)
test_eq((out3 / "adapter" / "adapter_config.json").exists(), True)
def probs(a): return list(a["probabilities"].values()) if "probabilities" in a else [a["noul"]]
a3 = Decider.load(out3, device="cpu").predict(rec["state"], rec["questions"])
b3 = ll.predict(rec["state"], rec["questions"])
for qid in a3: test_close(probs(a3[qid]), probs(b3[qid]), eps=1e-3)

A Laya checkpoint loads as a `Decider` directly. Copying only Laya's files out of the export makes a plain Laya checkpoint, and it answers exactly as the sysone export does:

In [ ]:
laya_only = Path(tempfile.mkdtemp()) / "laya-only"
shutil.copytree(out, laya_only, ignore=shutil.ignore_patterns("sysone.json", "head.safetensors"))
jl = Decider.load(laya_only, device="cpu")
test_eq(jl.name, "laya")
test_eq(jl.predict(rec["state"], rec["questions"]), jev.predict(rec["state"], rec["questions"]))

On real weights: Laya's published typed-decisions checkpoint, loaded by `Decider.load` (the budgets, 1,024 / 256, and nine temperatures come from its config) and scored on the test split with `sysone.evaluate`:

In [ ]:
#| eval: false
from datasets import load_dataset
from sysone.evaluate import evaluate
laya_td = Decider.load("convaiinnovations/laya/typed-decisions")
evaluate(laya_td, load_dataset("LocalLLaMA/typed-decisions", "all", split="test"), by="workflow")

The run, pasted (2026-09-26, M3 Pro, MPS, fp32):

```
accuracy 0.766 (choice 0.733, score 0.723, noul 0.857) on 2,000 decisions; soft accuracy 0.509, Brier 0.066, KL 0.122,
ECE 0.213, score MAE 0.242, within one level 0.995; latency p50 444 ms per case (five questions, one forward pass)
by workflow: agent_trace_observability 0.730, customer_service 0.764, invoice_processing 0.804, security_incidents 0.766
```

0.766 is the accuracy Laya's fine-tuning notebook reports for this checkpoint. On the first 40 test cases, 200 decisions, `laya.Agent` on the same checkpoint gives the same answer to every one, with probabilities within 0.002 of sysone's (float noise on MPS; the mean difference is 0.0002).

### Side streams and the act head in an export

An export of a model with side streams saves each stream's tokenizer under `streams/`, the stream encoder as transformers folders plus its own small file, and the head's whole shape: its readout, its query and its act head. The act threshold fitted by `calibrate` goes into `sysone.json`, and the loaded `Decider` reports `action` exactly as the learner's did. On the fixture, a stream of free-text notes stands in for a protein: the tiny ModernBERT reads the row, a tiny ESM reads the notes, and a pair head scores the options against the notes' vector:

In [ ]:
from transformers import AutoConfig, EsmConfig, EsmModel

In [ ]:
def notes_spec():
    "A spec whose encoder is the fixture's text encoder beside a tiny ESM over a `notes` stream"
    tk = AutoTokenizer.from_pretrained(FIX)
    def loader(spec, dtype=torch.float32, **kw):
        torch.manual_seed(0)
        side = EsmModel(EsmConfig(vocab_size=len(tk), hidden_size=32, num_hidden_layers=1, num_attention_heads=2, intermediate_size=64,
                                  max_position_embeddings=64, pad_token_id=tk.pad_token_id, position_embedding_type="absolute",
                                  token_dropout=False), add_pooling_layer=False)
        special = (tk.cls_token_id, tk.sep_token_id, tk.pad_token_id)
        return StreamEncoder(AutoModel.from_pretrained(FIX, dtype=dtype), {"notes": SideEncoder(side, special)}).to(dtype)
    return EncoderSpec(FIX, AutoConfig.from_pretrained(FIX), tk, loader=loader, streams=[TokenStream("notes", tk, max_len=24)])

srecs = [dict(r, state={"text": json.dumps(r["state"]), "notes": f"note {i}: the customer wrote {i % 3} times"}) for i, r in enumerate(recs)]
sdata = TypedDecisions.from_records(srecs, valid=0.25, calib=0.25, tfms=[Stream("notes")])
torch.manual_seed(0)
slearn = sysone.learner.decision_learner(sdata, notes_spec(), train="head", preset="cpu", loss="soft_ce", head="pair",
                                         query="stream:notes", act=True, calibrate_after_fit=False)
slearn.fit(1, lr=1e-3); slearn.calibrate(act_accuracy=0.5)
sq = srecs[1]
before = slearn.predict(sq["state"], sq["questions"])
sout = slearn.export(Path(tempfile.mkdtemp()) / "notes-jev")
sjev = Decider.load(sout, device="cpu")
test_eq(sjev.predict(sq["state"], sq["questions"]), before)
test_eq(sorted(p.name for p in (sout / "streams").iterdir()), ["notes"])
test_eq((sout / "encoder" / "stream_encoder.json").exists(), True)
test_eq(sjev.act_threshold, slearn.act_threshold)
next(iter(before.values()))

In [ ]:
slearn.calibrate(act_accuracy=0.5, act_score="confidence")          # the other rule, exported and loaded back
cjev = Decider.load(slearn.export(Path(tempfile.mkdtemp()) / "notes-conf"), device="cpu")
test_eq((cjev.act_score, cjev.predict(sq["state"], sq["questions"])), ("confidence", slearn.predict(sq["state"], sq["questions"])))

In [ ]:
test_eq("action" in next(iter(before.values())), True)

## ONNX

For serving a text model on CPUs without PyTorch, `export_onnx` writes an ordinary export plus `model.onnx`: the encoder and the head as one graph that returns a score for every position. The gather at the anchors and the softmax stay outside the graph, in numpy, so one graph serves any number of options. `OnnxDecider` runs it with onnxruntime (`sysone[onnx]`) and answers as the `Decider` does; it is a `Decider` whose logits come from the graph.

```{mermaid}
flowchart LR
    subgraph ONNX["model.onnx (any batch, any length)"]
        I["input_ids,<br/>attention_mask, qtype"] --> E["encoder"] --> H["head layers<br/>and scorer"] --> S["a score for every<br/>position (batch, L)"]
    end
    S --> GA["numpy: the scores<br/>at marker_pos"] --> SM["softmax over<br/>the options"]
```

Keeping the gather outside the graph is what lets one exported graph serve questions with any number of options: the graph never sees K.

In [ ]:
#| export
class PositionScores(torch.nn.Module):
    "The encoder and the head as one module that scores every position: the graph `export_onnx` writes"
    def __init__(self, model:EncoderDecisionModel): super().__init__(); self.m = model
    def forward(self, input_ids, attention_mask, qtype):
        head = self.m.head
        h = head._standardized(self.m.encode(input_ids, attention_mask).to(head.type_emb.weight.dtype)) + head.type_emb(qtype)[:, None, :]
        pad = ~attention_mask.bool()
        for blk in head.blocks: h = blk(h, src_key_padding_mask=pad)
        return head.scorer(h).squeeze(-1).float()

def export_onnx(learn:Learner, path, **kw) -> Path:
    "An export (see `export_learner`) with the model as `model.onnx` beside it"
    b, hd = learn.data.builder, learn.model.head
    if b.processor is not None or b.streams: raise ValueError("ONNX export covers text models; multimodal and side-stream ones run through PyTorch")
    if hd.act or hd.query is not None or hd.readout != "anchor":
        raise ValueError("ONNX export covers heads that score each anchor on its own; an act head, a query or a span readout runs through PyTorch")
    path = export_learner(learn, path, **kw)
    m = EncoderDecisionModel(copy.deepcopy(merged_encoder(learn.model)).cpu().float(), copy.deepcopy(learn.model.head).cpu().float()).eval()
    row = b.build("an example state", {"type": "choice", "instructions": "an example question", "criteria": ["a", "b", "c"]})
    x = collate([row, row], b.tok.pad_token_id or 0)
    B, L = torch.export.Dim("batch"), torch.export.Dim("length")
    shapes = {"input_ids": {0: B, 1: L}, "attention_mask": {0: B, 1: L}, "qtype": {0: B}}
    with warnings.catch_warnings():   # torch names a shared axis once, and warns for every other input that has it
        warnings.filterwarnings("ignore", message="# The axis name")
        torch.onnx.export(PositionScores(m).eval(), (x["input_ids"], x["attention_mask"], x["qtype"]), str(path / "model.onnx"),
                          input_names=["input_ids", "attention_mask", "qtype"], output_names=["scores"], dynamic_shapes=shapes,
                          dynamo=True, verbose=False)
    meta = json.loads((path / "sysone.json").read_text()); meta["onnx"] = "model.onnx"
    (path / "sysone.json").write_text(json.dumps(meta, indent=2, default=str))
    return path

class OnnxDecider(Decider):
    "A `Decider` whose logits come from an exported ONNX graph run by onnxruntime"
    def __init__(self, session, builder:RowBuilder, temperatures:dict|None=None, batch_size:int=64, digits:int|None=4,
                 name:str="sysone", chunk:int|None=None, meta:dict|None=None):
        self.session, self.builder, self.temperatures = session, builder, dict(temperatures or {})
        self.batch_size, self.digits, self.name, self.chunk, self.meta = batch_size, digits, name, chunk, meta
        self.act_threshold, self.act_score = None, "head"
        self.model, self.device, self.dtype = None, torch.device("cpu"), None
    def __repr__(self): return f"OnnxDecider({self.name} · onnxruntime · {len(self.temperatures)} temperatures)"

    def _scores(self, rows) -> tuple: return self._logits(rows), None
    def _logits(self, rows) -> np.ndarray:
        out, pad = [], self.builder.tok.pad_token_id or 0
        for i in range(0, len(rows), self.batch_size):
            b = collate(rows[i:i + self.batch_size], pad)
            s = self.session.run(None, {k: b[k].numpy() for k in ("input_ids", "attention_mask", "qtype")})[0]
            out.append(np.where(b["marker_mask"].numpy(), np.take_along_axis(s, b["marker_pos"].numpy(), 1), -1e4))
        K = max(z.shape[1] for z in out)
        return np.concatenate([np.pad(z, ((0, 0), (0, K - z.shape[1])), constant_values=-1e4) for z in out])

    @classmethod
    def load(cls, path, providers=None, **kw):
        "The ONNX decider of an `export_onnx` folder"
        import onnxruntime as ort
        path = Path(path); meta = json.loads((path / "sysone.json").read_text())
        builder = RowBuilder.from_json({"template": meta["template"], "tfms": meta.get("tfms", [])}, AutoTokenizer.from_pretrained(path / "tokenizer"))
        sess = ort.InferenceSession(str(path / meta.get("onnx", "model.onnx")), providers=providers or ["CPUExecutionProvider"])
        return cls(sess, builder, meta.get("temperature"), name=meta.get("name", "sysone"), meta=meta, **kw)

In [ ]:
test_fail(lambda: export_onnx(slearn, Path(tempfile.mkdtemp()) / "x"), contains="PyTorch")   # a side stream and an act head: PyTorch only

In [ ]:
out4 = export_onnx(learn, Path(tempfile.mkdtemp()) / "onnx-jev")
ox = OnnxDecider.load(out4)
for rc in recs[:6]:
    a, b = ox.predict(rc["state"], rc["questions"]), jev.predict(rc["state"], rc["questions"])
    for qid in a: test_close(probs(a[qid]), probs(b[qid]), eps=2e-4)
ox, (out4 / "model.onnx").stat().st_size

## Serving

`serve_app(decider)` is the forty-line FastAPI app: `POST /v1/systemone` takes the Jev request body (`state`, `questions`, an optional `model`) and answers `{model, answers, usage}`, and `GET /health` reports the model and device. Malformed bodies get 400 and unanswerable questions 422, as Laya's server does, and so does any other error in the request's content (a field a transform needs, an unreadable image). Images travel inline, as base64 data URLs: a path in `state.image` would make the server open files of its own choosing on a client's behalf, so paths are refused unless `allow_image_paths=True`. For Laya-format text exports, `laya[serve]` serves the same route; this app covers what Laya's runtimes cannot load (other templates, multimodal encoders). `sysone serve my-jev` runs it (needs `sysone[serve]`).

```{mermaid}
sequenceDiagram
    participant C as client
    participant A as serve_app (FastAPI)
    participant D as Decider
    C->>A: POST /v1/systemone {state, questions, model?}
    alt not JSON, no questions object, or no state
        A-->>C: 400
    else more than max_questions
        A-->>C: 413
    else an image given as a server path
        A-->>C: 422: send it inline as a data URL
    else
        A->>D: predict(state, questions)
        D-->>A: answers, or an error in the request's content (422)
        A-->>C: {model, answers, usage}
    end
```

In [ ]:
#| export
def _server_paths(state):
    "Image values of a state that are not inline data URLs"
    if not isinstance(state, dict): return []
    return [v for k, v in state.items() if k in ("image", "images") for v in (v if isinstance(v, list) else [v])
            if isinstance(v, str) and not v.startswith("data:image/")]

def serve_app(decider:Decider, max_questions:int=64, allow_image_paths:bool=False):
    "A FastAPI app answering `POST /v1/systemone` and `GET /health` with `decider`"
    from fastapi import FastAPI, HTTPException, Request
    app = FastAPI(title=f"sysone · {decider.name}")

    @app.get("/health")
    def health(): return {"status": "ok", "model": decider.name, "device": str(decider.device)}

    @app.post("/v1/systemone")
    async def systemone(request: Request):
        try: body = await request.json()
        except Exception: raise HTTPException(400, "request body must be valid JSON")
        if not isinstance(body, dict) or not isinstance(body.get("questions"), dict):
            raise HTTPException(400, "request body must be an object with a 'questions' object")
        if len(body["questions"]) > max_questions: raise HTTPException(413, f"too many questions ({len(body['questions'])} > {max_questions})")
        if body.get("state") is None: raise HTTPException(400, "'state' is required")
        if not allow_image_paths and _server_paths(body["state"]):
            raise HTTPException(422, "send images inline as data URLs (data:image/png;base64,...); server paths are not read")
        try: answers, n = decider.answer(body["state"], body["questions"])
        except (ValueError, KeyError, TypeError, OSError) as e:   # the request's content: a question, a field, an image
            raise HTTPException(422, f"{type(e).__name__}: {e}")
        return {"model": decider.name, "answers": answers, "usage": {"input_tokens": n, "output_tokens": 0}}
    return app

In [ ]:
from fastapi.testclient import TestClient

In [ ]:
client = TestClient(serve_app(jev))
r = client.post("/v1/systemone", json={"state": rec["state"], "questions": rec["questions"], "model": "jev-latest"})
test_eq(r.status_code, 200)
test_eq(r.json()["answers"], json.loads(json.dumps(jev.predict(rec["state"], rec["questions"]))))
test_eq(client.post("/v1/systemone", json={"state": "x", "questions": {"q": {"type": "rank", "instructions": "?"}}}).status_code, 422)
test_eq(client.post("/v1/systemone", content=b"not json").status_code, 400)
test_eq(client.post("/v1/systemone", json={"state": {"image": "/etc/passwd", "text": "x"}, "questions": rec["questions"]}).status_code, 422)
client.get("/health").json(), r.json()["usage"]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()